## Downloading necessary libraries

In [ ]:
!pip install sentence-transformers pandas

## Training the Model

In [2]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
import math
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# In Colab, uploaded files go straight to the /content/ directory
data_path = "/content/training_pairs.csv"
model_save_path = "/content/sbert_finetuned"

logging.info("Checking hardware...")
device = "cuda" if torch.cuda.is_available() else "cpu"
logging.info(f"Executing on: {device.upper()}")
if device == "cpu":
    logging.warning("WARNING: You didn't select a T4 GPU in the runtime settings! Stop the cell and change it.")

logging.info("Loading dataset...")
df = pd.read_csv(data_path)

logging.info("Downloading base all-MiniLM-L6-v2 model...")
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

logging.info("Formatting tensor pairs...")
train_examples = []

for _, row in df.iterrows():
    resume_input = f"SKILLS: {row['resume_skills']} | CONTEXT: {str(row['resume_text'])[:1000]}"
    jd_input = f"SKILLS: {row['jd_skills']} | CONTEXT: {str(row['jd_text'])[:1000]}"
    label = float(row['label'])

    train_examples.append(InputExample(texts=[resume_input, jd_input], label=label))

# 90/10 Train/Validation Split
split_idx = int(0.9 * len(train_examples))
train_data = train_examples[:split_idx]
val_data = train_examples[split_idx:]

train_dataloader = DataLoader(train_data, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(model)

sentences1 = [ex.texts[0] for ex in val_data]
sentences2 = [ex.texts[1] for ex in val_data]
scores = [ex.label for ex in val_data]
evaluator = evaluation.EmbeddingSimilarityEvaluator(sentences1, sentences2, scores)

num_epochs = 4
warmup_steps = math.ceil(len(train_dataloader) * num_epochs * 0.1)

logging.info(f"Starting Training: {num_epochs} Epochs, {len(train_data)} Pairs.")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=num_epochs,
    evaluation_steps=500,
    warmup_steps=warmup_steps,
    output_path=model_save_path
)

logging.info(f"✅ Training Complete. Model saved to {model_save_path}")

/tmp/ipykernel_1380/740913160.py:4: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
/tmp/ipykernel_1380/740913160.py:4: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Pearson Cosine,Spearman Cosine
254,No log,No log,0.842144,0.860698
500,0.067377,No log,0.863984,0.872290
508,0.067377,No log,0.858155,0.867642
762,0.067377,No log,0.862231,0.875431
1000,0.040445,No log,0.860263,0.870368
1016,0.040445,No log,0.860125,0.870411


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Saving the Model

In [3]:
import shutil
from google.colab import files

print("Zipping model directory...")
shutil.make_archive("/content/sbert_finetuned", 'zip', "/content/sbert_finetuned")

print("Triggering download...")
files.download("/content/sbert_finetuned.zip")

Zipping model directory...
Triggering download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>